# 🧠 EEG Preprocessing Pipeline — Stage 3: ICA Label + Apply

**Dataset:** `ds006780` (ASD EEG, BIDS-formatted)  
**Inputs (read from GCS):**
- Stage 1: `desc-preproc_eeg.fif` — filtered, resampled, bads interpolated, **NOT yet re-referenced**
- Stage 2: `desc-preproc_ica.fif` — ICA decomposition, fit on average-referenced data

**Outputs (written to GCS):**
- `desc-clean_eeg.fif` — cleaned EEG (avg ref + bad ICs removed)
- `desc-iclabel_components.tsv` — per-component ICLabel labels + class probabilities

---

## What this stage does

```
 Stage 1 raw  +  Stage 2 ICA
      │              │
      ▼              ▼
 avg reference  ←  must match Stage 2's pre-ICA reference
      │
      ▼
 ICLabel  →  per-IC label + probability  →  save TSV
      │
      ▼
 ica.exclude = bad ICs
      │
      ▼
 ica.apply(raw)  →  desc-clean_eeg.fif
```

## ⚠️ Critical: re-applying average reference

Stage 1 saves the raw **before** average referencing.  
Stage 2 applies avg ref *in memory*, fits ICA, but only saves the decomposition.

So in Stage 3 we **must** re-apply avg ref to the Stage 1 raw before `ica.apply()` —  
otherwise the mixing matrix is referenced to a different signal than the data and the result is garbage.

## Why bundle label + apply

Both depend on the same `(raw, ica)` pair. If you later want stricter rejection,  
re-run Stage 3 only — Stage 2's expensive decomposition stays untouched.

## Alignment with prior stages

| Concept            | Stage 1                    | Stage 2                    | Stage 3                     |
|---                 |---                         |---                         |---                          |
| Pipeline name      | `mne-preproc-pre-ica`      | `mne-ica-fit`              | `mne-ica-apply`             |
| Output entity      | `desc-preproc_eeg.fif`     | `desc-preproc_ica.fif`     | `desc-clean_eeg.fif` + TSV  |
| Resume logic       | skip if output exists      | skip if output exists      | skip if both outputs exist  |
| Smoke test         | yes                        | yes                        | yes                         |

---
# 1. Setup

## 1.1 Install dependencies (run once)

- `mne-icalabel` is the automated component classifier (runs ONNX inference, very fast)
- `onnxruntime` is the execution backend ICLabel uses
- `google-cloud-storage` for GCS I/O

In [25]:
!pip install mne mne-bids mne-icalabel pandas numpy joblib tqdm \
             google-cloud-storage onnxruntime


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## 1.2 Imports + sanity check

If `mne-icalabel` isn't installed we fail fast — there's no point continuing without it.

In [26]:
import gc
import json
import time
import traceback
from pathlib import Path
from datetime import datetime
from contextlib import contextmanager, nullcontext

import numpy as np
import pandas as pd
import mne
from mne.preprocessing import read_ica
from joblib import Parallel, delayed
from tqdm.auto import tqdm

# ICLabel
try:
    from mne_icalabel import label_components
    HAS_ICALABEL = True
except ImportError:
    HAS_ICALABEL = False

# GCS
try:
    from google.cloud import storage as gcs_lib
    from google.cloud.storage.retry import DEFAULT_RETRY
    HAS_GCS = True
except ImportError:
    HAS_GCS = False

mne.set_log_level('WARNING')
print(f'MNE version:         {mne.__version__}')
print(f'mne-icalabel ready:  {HAS_ICALABEL}')
print(f'GCS available:       {HAS_GCS}')

if not HAS_ICALABEL:
    raise RuntimeError(
        'mne-icalabel not installed. Run: pip install mne-icalabel onnxruntime'
    )

MNE version:         1.10.2
mne-icalabel ready:  True
GCS available:       True


---
# 2. Configuration

## 2.1 Paths

Stage 3 reads from **two** prior derivatives (Stage 1 raw, Stage 2 ICA) and writes a new derivative dataset (`mne-ica-apply`).

In [27]:
PROJECT_ROOT     = Path.home() / 'asd_eeg_pipeline'
DATASET_ID       = 'ds006780'

STAGE1_PIPELINE  = 'mne-preproc-pre-ica'      # raw input (preprocessed EEG)
STAGE2_PIPELINE  = 'mne-ica-fit'              # ICA decomposition input
PIPELINE_NAME    = 'mne-ica-apply'            # Stage 3 output

STAGE1_LOCAL     = PROJECT_ROOT / 'derivatives' / STAGE1_PIPELINE
STAGE2_LOCAL     = PROJECT_ROOT / 'derivatives' / STAGE2_PIPELINE
DERIV_ROOT       = PROJECT_ROOT / 'derivatives' / PIPELINE_NAME
LOG_DIR          = PROJECT_ROOT / 'derivatives' / 'logs'

for d in (DERIV_ROOT, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'📤 Output (Stage 3):  {DERIV_ROOT}')

📤 Output (Stage 3):  /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-ica-apply


## 2.2 GCS configuration

This stage **reads from GCS and writes to GCS**.  
Local mode is also supported by flipping `INPUT_SOURCE = 'local'` and `UPLOAD_TO_GCS = False`.

Inputs are downloaded into a temp cache, used, and deleted — no permanent disk usage from GCS streaming.

In [28]:
INPUT_SOURCE  = 'gcs'    # 'gcs' or 'local'
UPLOAD_TO_GCS = True     # upload Stage 3 outputs to GCS

GCS_BUCKET           = 'asd-eeg-dataset'
GCS_STAGE1_PREFIX    = f'derivatives/{DATASET_ID}/{STAGE1_PIPELINE}'
GCS_STAGE2_PREFIX    = f'derivatives/{DATASET_ID}/{STAGE2_PIPELINE}'
GCS_STAGE3_PREFIX    = f'derivatives/{DATASET_ID}/{PIPELINE_NAME}'

# Local cache for streamed inputs (cleaned up after each file)
INPUT_CACHE = Path('/tmp/stage3_cache')
INPUT_CACHE.mkdir(parents=True, exist_ok=True)

print(f'INPUT_SOURCE   = {INPUT_SOURCE}')
print(f'UPLOAD_TO_GCS  = {UPLOAD_TO_GCS}')
print(f'GCS Stage 1    = gs://{GCS_BUCKET}/{GCS_STAGE1_PREFIX}/')
print(f'GCS Stage 2    = gs://{GCS_BUCKET}/{GCS_STAGE2_PREFIX}/')
print(f'GCS Stage 3    = gs://{GCS_BUCKET}/{GCS_STAGE3_PREFIX}/')

INPUT_SOURCE   = gcs
UPLOAD_TO_GCS  = True
GCS Stage 1    = gs://asd-eeg-dataset/derivatives/ds006780/mne-preproc-pre-ica/
GCS Stage 2    = gs://asd-eeg-dataset/derivatives/ds006780/mne-ica-fit/
GCS Stage 3    = gs://asd-eeg-dataset/derivatives/ds006780/mne-ica-apply/


Initialise the GCS client lazily — only construct it if we actually need it.  
The retry config is the same shape as your Stage 2: deadlined retries with exponential backoff, generous timeouts and 16 MB chunks for fast uploads.

In [29]:
gcs_client  = None
gcs_bucket  = None
GCS_RETRY   = None
GCS_TIMEOUT = 600
GCS_CHUNK   = 16 * 1024 * 1024

if INPUT_SOURCE == 'gcs' or UPLOAD_TO_GCS:
    if not HAS_GCS:
        raise RuntimeError(
            'google-cloud-storage is not installed. '
            'Run: pip install google-cloud-storage'
        )
    gcs_client  = gcs_lib.Client()
    gcs_bucket  = gcs_client.bucket(GCS_BUCKET)
    GCS_RETRY   = DEFAULT_RETRY.with_deadline(600).with_delay(
                      initial=2, maximum=30, multiplier=2)
    print(f'✅ GCS client ready  →  gs://{GCS_BUCKET}/')
else:
    print('ℹ️  GCS skipped (INPUT_SOURCE=local, UPLOAD_TO_GCS=False)')

✅ GCS client ready  →  gs://asd-eeg-dataset/


## 2.3 ICLabel + rejection policy

ICLabel returns one of 7 labels per component, plus the full probability vector.  
The 7 classes are: `brain`, `muscle artifact`, `eye blink`, `heart beat`, `line noise`, `channel noise`, `other`.

**Default policy:**
- KEEP: `brain`, `other` (mixed/uncertain — safer to keep)
- DROP: everything else, **but only if the predicted-label probability ≥ 0.80**

The probability threshold is the conservative knob — at 0.0 you trust the predicted label outright (more aggressive cleaning), at 0.95 you only drop components ICLabel is very sure are bad (more conservative).

If a single file gets >50% of components dropped, that's a strong signal something went wrong with the ICA fit for that recording — we flag it in the log.

In [30]:
EXCLUDE_LABELS = {
    'muscle artifact',
    'eye blink',
    'heart beat',
    'line noise',
    'channel noise',
}

# Minimum ICLabel probability for a component to actually be dropped
LABEL_PROB_THRESHOLD = 0.80

# Flag (don't fail) files where more than this fraction of ICs get dropped
MAX_BAD_FRACTION_WARN = 0.50

print(f'DROP labels:               {sorted(EXCLUDE_LABELS)}')
print(f'Probability threshold:     {LABEL_PROB_THRESHOLD}')
print(f'High-bad-fraction warning: > {MAX_BAD_FRACTION_WARN:.0%}')

DROP labels:               ['channel noise', 'eye blink', 'heart beat', 'line noise', 'muscle artifact']
Probability threshold:     0.8
High-bad-fraction warning: > 50%


## 2.4 Tasks and execution

ICLabel is fast (<5 s/file), so most of the wall time is `ica.apply()` and I/O.  
Bumping `N_JOBS_OUTER` helps if you have spare RAM, but watch memory because each worker  
holds a Raw and an ICA in memory simultaneously.

In [31]:
TASKS        = ['Restingstate', 'FAST', 'IC', 'motor']   # or None for all
N_JOBS_OUTER = 1                                          # parallel files
OVERWRITE    = False                                      # redo finished files?

print(f'TASKS        = {TASKS}')
print(f'N_JOBS_OUTER = {N_JOBS_OUTER}')
print(f'OVERWRITE    = {OVERWRITE}')

TASKS        = ['Restingstate', 'FAST', 'IC', 'motor']
N_JOBS_OUTER = 1
OVERWRITE    = False


---
# 3. Initialize BIDS-derivatives dataset

Same pattern as Stage 2 — write a `dataset_description.json` so this output is itself a valid BIDS-derivative dataset that points to **both** of its source datasets.

In [32]:
def init_bids_derivatives(deriv_root, pipeline_name):
    desc = {
        'Name': f'{DATASET_ID} — {pipeline_name}',
        'BIDSVersion': '1.9.0',
        'DatasetType': 'derivative',
        'GeneratedBy': [{
            'Name': pipeline_name,
            'Version': '0.1.0',
            'Description': (
                f'ICA component labeling (ICLabel) and application. '
                f'Drop labels: {sorted(EXCLUDE_LABELS)} '
                f'with probability >= {LABEL_PROB_THRESHOLD}. '
                f'Output is average-referenced, ICA-cleaned EEG.'
            ),
            'CodeURL': 'local notebook',
        }],
        'SourceDatasets': [
            {'URL': f'derivatives/{STAGE1_PIPELINE}'},
            {'URL': f'derivatives/{STAGE2_PIPELINE}'},
        ],
    }
    out = Path(deriv_root) / 'dataset_description.json'
    out.write_text(json.dumps(desc, indent=2))
    return out

desc_path = init_bids_derivatives(DERIV_ROOT, PIPELINE_NAME)
print(f'✅ Wrote {desc_path}')

✅ Wrote /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-ica-apply/dataset_description.json


---
# 4. Core pipeline functions

Same modular pattern as Stage 1 and 2 — small single-purpose functions, with `process_one` as the orchestrator at the end.

## 4.1 BIDS filename parser

Same parser as Stage 2, plus a `make_key` helper that gives us a stable tuple `(subject, task, run)` for matching Stage 1 files to Stage 2 files.

In [33]:
def parse_bids_filename(fname):
    """Pull subject/task/run/desc out of a BIDS filename."""
    parts = Path(fname).stem.split('_')
    fields = {}
    for p in parts:
        if '-' in p:
            key, val = p.split('-', 1)
            fields[key] = val
    return {
        'subject': fields.get('sub'),
        'task':    fields.get('task'),
        'run':     fields.get('run'),
        'desc':    fields.get('desc'),
    }


def make_key(entities):
    """Stable key for matching Stage 1 raw to Stage 2 ICA."""
    return (entities['subject'], entities['task'], entities.get('run'))

# Quick sanity check
demo = parse_bids_filename('sub-10003_task-FAST_run-01_desc-preproc_eeg.fif')
print(demo)
print('key:', make_key(demo))

{'subject': '10003', 'task': 'FAST', 'run': '01', 'desc': 'preproc'}
key: ('10003', 'FAST', '01')


## 4.2 Discovery — pair Stage 1 raw with Stage 2 ICA

Each Stage 3 input is a triple `(raw_ref, ica_ref, entities)`. The references are either local `Path`s or GCS blob names depending on `INPUT_SOURCE`.

If a raw file has no matching ICA, we report it but skip — that recording can't be applied without its decomposition.

In [34]:
def discover_local(stage1_root, stage2_root, tasks=None):
    """Find (raw_path, ica_path, entities) triples on local disk."""
    raw_files = sorted(Path(stage1_root).rglob('*_desc-preproc_eeg.fif'))
    ica_files = sorted(Path(stage2_root).rglob('*_desc-preproc_ica.fif'))

    ica_by_key = {make_key(parse_bids_filename(p.name)): p for p in ica_files}

    out, missing_ica = [], []
    for r in raw_files:
        ent = parse_bids_filename(r.name)
        if tasks is not None and ent['task'] not in tasks:
            continue
        key = make_key(ent)
        if key not in ica_by_key:
            missing_ica.append(r.name)
            continue
        out.append((r, ica_by_key[key], ent))
    return out, missing_ica

In [35]:
def discover_gcs(stage1_prefix, stage2_prefix, tasks=None):
    """Find (raw_blob, ica_blob, entities) triples in GCS."""
    raw_blobs, ica_blobs = [], []
    for blob in gcs_client.list_blobs(GCS_BUCKET, prefix=f'{stage1_prefix}/'):
        if blob.name.endswith('_desc-preproc_eeg.fif'):
            raw_blobs.append(blob.name)
    for blob in gcs_client.list_blobs(GCS_BUCKET, prefix=f'{stage2_prefix}/'):
        if blob.name.endswith('_desc-preproc_ica.fif'):
            ica_blobs.append(blob.name)

    ica_by_key = {
        make_key(parse_bids_filename(Path(b).name)): b for b in ica_blobs
    }

    out, missing_ica = [], []
    for rb in sorted(raw_blobs):
        ent = parse_bids_filename(Path(rb).name)
        if tasks is not None and ent['task'] not in tasks:
            continue
        key = make_key(ent)
        if key not in ica_by_key:
            missing_ica.append(Path(rb).name)
            continue
        out.append((rb, ica_by_key[key], ent))
    return out, missing_ica

## 4.3 GCS staging helpers

- `staged_from_gcs` is a context manager: download → yield local path → delete.  
  This means cache disk usage stays bounded by `(N_JOBS_OUTER × 2)` files at most.
- `upload_to_gcs` is **idempotent**: skip the upload if a blob of identical size already exists at the destination. Lets you re-run the upload step cheaply.

In [36]:
@contextmanager
def staged_from_gcs(blob_name, cache_dir=INPUT_CACHE, cleanup=True):
    """Download blob to cache, yield local path, delete after."""
    local_path = cache_dir / Path(blob_name).name
    if not local_path.exists():
        blob = gcs_bucket.blob(blob_name, chunk_size=GCS_CHUNK)
        blob.download_to_filename(str(local_path), timeout=GCS_TIMEOUT, retry=GCS_RETRY)
    try:
        yield local_path
    finally:
        if cleanup:
            try:
                local_path.unlink(missing_ok=True)
            except OSError:
                pass


def upload_to_gcs(local_path, gcs_prefix=GCS_STAGE3_PREFIX):
    """Upload Stage 3 output to GCS. Skips if size matches existing blob."""
    rel = local_path.relative_to(DERIV_ROOT)
    blob_name = f'{gcs_prefix}/{rel.as_posix()}'
    blob = gcs_bucket.blob(blob_name, chunk_size=GCS_CHUNK)
    if blob.exists(retry=GCS_RETRY, timeout=GCS_TIMEOUT):
        blob.reload(retry=GCS_RETRY, timeout=GCS_TIMEOUT)
        if blob.size == local_path.stat().st_size:
            return f'gs://{GCS_BUCKET}/{blob_name}'
    blob.upload_from_filename(str(local_path), timeout=GCS_TIMEOUT, retry=GCS_RETRY)
    return f'gs://{GCS_BUCKET}/{blob_name}'

## 4.4 Loading + average reference

`apply_average_reference` is **intentionally identical** to Stage 2's function — same operation must produce identical pre-ICA data, otherwise the mixing matrix doesn't match the input.

In [37]:
def load_raw(fpath):
    return mne.io.read_raw_fif(fpath, preload=True, verbose=False)


def load_ica(fpath):
    return read_ica(fpath, verbose=False)


def apply_average_reference(raw):
    """Same as Stage 2 — must match exactly."""
    raw.set_eeg_reference('average', projection=True, verbose=False)
    raw.apply_proj(verbose=False)
    return raw

## 4.5 ICLabel — classify components

`mne_icalabel.label_components(raw, ica, method='iclabel')` returns:
- `labels` — predicted class string per component
- `y_pred_proba` — full probability matrix `(n_components, 7)`

We turn that into a **DataFrame with one row per component**, including all 7 class probabilities. This is what gets saved as the BIDS-style `_components.tsv`. The `exclude` column already encodes our policy (label-in-excludes ∧ probability ≥ threshold), so downstream code doesn't need to re-derive it.

In [38]:
# Use the lower-level function that returns the FULL (n_components, 7)
# probability matrix. The high-level `label_components` only returns
# the winning probability per component (1D), which is not enough
# to populate per-class columns in the TSV.
from mne_icalabel.iclabel import iclabel_label_components

# ICLabel class order — DO NOT change this; it's fixed by the model.
ICLABEL_CLASSES = [
    'brain', 'muscle artifact', 'eye blink', 'heart beat',
    'line noise', 'channel noise', 'other',
]


def label_ics(raw, ica):
    """Run ICLabel; return DataFrame with one row per component.

    Each row carries: predicted label, winning probability, and the
    full per-class probability vector — useful for re-thresholding
    later without re-running ICLabel.
    """
    # Full probability matrix: shape (n_components, 7)
    proba_matrix = iclabel_label_components(raw, ica)
    proba_matrix = np.asarray(proba_matrix)

    # Predicted label per component is just argmax across the 7 classes
    pred_idx      = np.argmax(proba_matrix, axis=1)
    labels        = [ICLABEL_CLASSES[i] for i in pred_idx]
    winning_probs = proba_matrix[np.arange(len(pred_idx)), pred_idx]

    rows = []
    for i, (lbl, p_win, p_row) in enumerate(
            zip(labels, winning_probs, proba_matrix)):
        exclude = (lbl in EXCLUDE_LABELS) and (float(p_win) >= LABEL_PROB_THRESHOLD)
        rows.append({
            'component':         i,
            'label':             lbl,
            'label_probability': round(float(p_win), 4),
            'prob_brain':         round(float(p_row[0]), 4),
            'prob_muscle':        round(float(p_row[1]), 4),
            'prob_eye':           round(float(p_row[2]), 4),
            'prob_heart':         round(float(p_row[3]), 4),
            'prob_line_noise':    round(float(p_row[4]), 4),
            'prob_channel_noise': round(float(p_row[5]), 4),
            'prob_other':         round(float(p_row[6]), 4),
            'exclude':            bool(exclude),
        })
    return pd.DataFrame(rows)

## 4.6 Output paths and savers

BIDS-derivative layout — same shape as your prior stages:

```
derivatives/mne-ica-apply/sub-XXX/eeg/
  sub-XXX_task-Y_run-ZZ_desc-clean_eeg.fif
  sub-XXX_task-Y_run-ZZ_desc-iclabel_components.tsv
```

In [39]:
def stage3_output_dir(entities, deriv_root=DERIV_ROOT):
    return Path(deriv_root) / f'sub-{entities["subject"]}' / 'eeg'


def stage3_basename(entities):
    parts = [f'sub-{entities["subject"]}', f'task-{entities["task"]}']
    if entities.get('run'):
        parts.append(f'run-{entities["run"]}')
    return '_'.join(parts)


def cleaned_eeg_path(entities, deriv_root=DERIV_ROOT):
    out_dir = stage3_output_dir(entities, deriv_root)
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / f'{stage3_basename(entities)}_desc-clean_eeg.fif'


def labels_tsv_path(entities, deriv_root=DERIV_ROOT):
    out_dir = stage3_output_dir(entities, deriv_root)
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / f'{stage3_basename(entities)}_desc-iclabel_components.tsv'


def save_cleaned_raw(raw, entities, deriv_root=DERIV_ROOT):
    out = cleaned_eeg_path(entities, deriv_root)
    raw.save(out, overwrite=True, verbose=False)
    return out


def save_labels_tsv(df, entities, deriv_root=DERIV_ROOT):
    out = labels_tsv_path(entities, deriv_root)
    df.to_csv(out, sep='\t', index=False)
    return out

## 4.7 Logging — fixed-schema CSV

**This fixes a bug in your Stage 2 log.** In Stage 2, `append_log` writes the CSV header from the first row's keys. If the first row is a `skipped` record (4 fields) and later rows are `ok` records (12+ fields), pandas emits inconsistent line widths and `pd.read_csv` later fails with `ParserError: Expected 6 fields in line 3, saw 13`.

**Fix:** pin the column schema upfront, pad every row to it. Now every row has the same width regardless of status.

In [40]:
LOG_FILE  = LOG_DIR / '03_ica_label_apply_log.csv'
ERROR_LOG = LOG_DIR / '03_ica_label_apply_errors.txt'

LOG_COLUMNS = [
    'timestamp', 'subject', 'task', 'run', 'status',
    'n_components', 'n_excluded', 'frac_excluded',
    'n_brain', 'n_other', 'n_muscle', 'n_eye',
    'n_heart', 'n_line', 'n_channel',
    'mean_label_prob', 'apply_seconds',
    'cleaned_path', 'labels_path',
    'cleaned_gcs', 'labels_gcs',
    'warning', 'error',
]


def append_log(record, log_path=LOG_FILE):
    """Append one row, padded to LOG_COLUMNS."""
    record = {**record, 'timestamp': datetime.now().isoformat(timespec='seconds')}
    row = {col: record.get(col, '') for col in LOG_COLUMNS}
    df = pd.DataFrame([row], columns=LOG_COLUMNS)
    header = not Path(log_path).exists()
    df.to_csv(log_path, mode='a', header=header, index=False)

print(f'Log file:   {LOG_FILE}')
print(f'Error log:  {ERROR_LOG}')
print(f'Columns:    {len(LOG_COLUMNS)}')

Log file:   /Users/alirezafatemi/asd_eeg_pipeline/derivatives/logs/03_ica_label_apply_log.csv
Error log:  /Users/alirezafatemi/asd_eeg_pipeline/derivatives/logs/03_ica_label_apply_errors.txt
Columns:    23


## 4.8 The orchestrator — `process_one`

Per-file pipeline:  
**stage inputs → load → avg ref → label → apply → save → upload**

Critical details:
- Resume logic: if **both** outputs already exist and `OVERWRITE=False`, skip.
- The `with raw_cm as raw_path, ica_cm as ica_path` block guarantees GCS-cached files are deleted after use, even on failure.
- The `finally` block frees `raw` and `ica` before the next file — `ica.apply()` peaks at ~3× raw size, so cleanup matters.
- All errors are caught and traceback'd to `ERROR_LOG`. The batch never crashes on a single bad file.

In [41]:
def process_one(item, deriv_root=DERIV_ROOT, log_file=LOG_FILE):
    """
    item: (raw_ref, ica_ref, entities)
      refs are local Paths or GCS blob names depending on INPUT_SOURCE.
    """
    raw_ref, ica_ref, entities = item
    base = {k: entities.get(k) for k in ('subject', 'task', 'run')}
    out_clean  = cleaned_eeg_path(entities, deriv_root)
    out_labels = labels_tsv_path(entities, deriv_root)

    if not OVERWRITE and out_clean.exists() and out_labels.exists():
        result = {
            **base, 'status': 'skipped',
            'cleaned_path': str(out_clean),
            'labels_path':  str(out_labels),
        }
        if log_file is not None:
            append_log(result, log_file)
        return result

    raw = ica = None
    try:
        # Stage inputs
        if INPUT_SOURCE == 'gcs':
            raw_cm = staged_from_gcs(raw_ref)
            ica_cm = staged_from_gcs(ica_ref)
        else:
            raw_cm = nullcontext(raw_ref)
            ica_cm = nullcontext(ica_ref)

        with raw_cm as raw_path, ica_cm as ica_path:
            raw = load_raw(raw_path)
            ica = load_ica(ica_path)

            # Re-apply avg reference (must match Stage 2's pre-ICA state)
            raw = apply_average_reference(raw)

            # Label
            labels_df = label_ics(raw, ica)

            # Apply
            bad_ics = labels_df.loc[labels_df['exclude'], 'component'].tolist()
            ica.exclude = bad_ics

            t0 = time.time()
            ica.apply(raw, verbose=False)
            apply_seconds = round(time.time() - t0, 1)

            # Save
            save_cleaned_raw(raw, entities, deriv_root)
            save_labels_tsv(labels_df, entities, deriv_root)

        # Upload
        cleaned_gcs = labels_gcs = ''
        if UPLOAD_TO_GCS:
            cleaned_gcs = upload_to_gcs(out_clean)
            labels_gcs  = upload_to_gcs(out_labels)

        # Stats
        n_total = len(labels_df)
        n_bad   = len(bad_ics)
        frac    = n_bad / n_total if n_total else 0.0
        counts  = labels_df['label'].value_counts().to_dict()

        warning = ''
        if frac > MAX_BAD_FRACTION_WARN:
            warning = f'high_bad_fraction:{frac:.2f}'

        result = {
            **base,
            'status':            'ok',
            'n_components':      n_total,
            'n_excluded':        n_bad,
            'frac_excluded':     round(frac, 3),
            'n_brain':           counts.get('brain', 0),
            'n_other':           counts.get('other', 0),
            'n_muscle':          counts.get('muscle artifact', 0),
            'n_eye':             counts.get('eye blink', 0),
            'n_heart':           counts.get('heart beat', 0),
            'n_line':            counts.get('line noise', 0),
            'n_channel':         counts.get('channel noise', 0),
            'mean_label_prob':   round(float(labels_df['label_probability'].mean()), 3),
            'apply_seconds':     apply_seconds,
            'cleaned_path':      str(out_clean),
            'labels_path':       str(out_labels),
            'cleaned_gcs':       cleaned_gcs,
            'labels_gcs':        labels_gcs,
            'warning':           warning,
        }

    except Exception as e:
        result = {**base, 'status': 'fail', 'error': str(e)}
        with open(ERROR_LOG, 'a') as f:
            f.write(f'\n=== {entities} ===\n{traceback.format_exc()}\n')
    finally:
        if raw is not None:
            del raw
        if ica is not None:
            del ica
        gc.collect()

    if log_file is not None:
        append_log(result, log_file)
    return result

---
# 5. Find input files

List both Stage 1 and Stage 2 outputs (from GCS or disk) and pair them.  
If any raw file has no matching ICA, it's reported and skipped — that's almost always a sign Stage 2 failed for that file.

In [42]:
if INPUT_SOURCE == 'gcs':
    inputs, missing_ica = discover_gcs(GCS_STAGE1_PREFIX, GCS_STAGE2_PREFIX, tasks=TASKS)
    print(f'Found {len(inputs)} (raw, ica) pairs in GCS')
else:
    inputs, missing_ica = discover_local(STAGE1_LOCAL, STAGE2_LOCAL, tasks=TASKS)
    print(f'Found {len(inputs)} (raw, ica) pairs locally')

if missing_ica:
    print(f'\n⚠️  {len(missing_ica)} raw files have no matching ICA — they will be skipped.')
    for n in missing_ica[:5]:
        print(f'    - {n}')
    if len(missing_ica) > 5:
        print(f'    ... and {len(missing_ica) - 5} more')

subjects   = {ent['subject'] for _, _, ent in inputs}
tasks_seen = {ent['task']    for _, _, ent in inputs}
print(f'\n  Subjects: {len(subjects)}')
print(f'  Tasks:    {sorted(tasks_seen)}')
for ref, _, _ in inputs[:3]:
    name = ref.name if hasattr(ref, 'name') else Path(ref).name
    print(f'  {name}')

Found 2356 (raw, ica) pairs in GCS

⚠️  3 raw files have no matching ICA — they will be skipped.
    - sub-11426_task-motor_run-04_desc-preproc_eeg.fif
    - sub-11426_task-motor_run-07_desc-preproc_eeg.fif
    - sub-11426_task-motor_run-10_desc-preproc_eeg.fif

  Subjects: 136
  Tasks:    ['FAST', 'IC', 'Restingstate', 'motor']
  sub-10025_task-Restingstate_run-01_desc-preproc_eeg.fif
  sub-10025_task-Restingstate_run-02_desc-preproc_eeg.fif
  sub-10025_task-Restingstate_run-03_desc-preproc_eeg.fif


---
# 6. Smoke test on one file

**Always run this before launching the batch.** ICLabel + apply takes ~10–30 s per file, but if your config or paths are wrong you want to know now, not after 1500 files.

Steps:
1. Run `process_one` on the first input (without writing to the log).
2. Inspect the labels TSV — check the per-component classifications look reasonable.
3. Inspect the cleaned EEG file — verify it loads, has the right channels and duration.

In [43]:
test_item = inputs[0]
raw_ref, ica_ref, ent = test_item
raw_name = raw_ref.name if hasattr(raw_ref, 'name') else Path(raw_ref).name
ica_name = ica_ref.name if hasattr(ica_ref, 'name') else Path(ica_ref).name

print(f'🧪 Testing on:')
print(f'   raw: {raw_name}')
print(f'   ica: {ica_name}\n')

test_result = process_one(test_item, log_file=None)
print(json.dumps(test_result, indent=2, default=str))

🧪 Testing on:
   raw: sub-10025_task-Restingstate_run-01_desc-preproc_eeg.fif
   ica: sub-10025_task-Restingstate_run-01_desc-preproc_ica.fif

{
  "subject": "10025",
  "task": "Restingstate",
  "run": "01",
  "status": "skipped",
  "cleaned_path": "/Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-ica-apply/sub-10025/eeg/sub-10025_task-Restingstate_run-01_desc-clean_eeg.fif",
  "labels_path": "/Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-ica-apply/sub-10025/eeg/sub-10025_task-Restingstate_run-01_desc-iclabel_components.tsv"
}


Inspect the **labels TSV** for the smoke-test file. You should see ~60 components (matching Stage 2's `n_components_fit`), with `brain` being the most common label and a sprinkling of artifact labels.

In [44]:
test_out_labels = labels_tsv_path(test_item[2])
if test_out_labels.exists():
    smoke_df = pd.read_csv(test_out_labels, sep='\t')
    print(f'✅ Labels TSV: {test_out_labels}')
    print(f'   Components:   {len(smoke_df)}')
    print(f'   Excluded:     {smoke_df["exclude"].sum()}')
    print()
    print('Label counts:')
    print(smoke_df['label'].value_counts().to_string())
    print()
    print('First 10 components:')
    print(smoke_df[['component', 'label', 'label_probability', 'exclude']]
          .head(10).to_string(index=False))
else:
    print(f'❌ No labels TSV — check {ERROR_LOG}')

✅ Labels TSV: /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-ica-apply/sub-10025/eeg/sub-10025_task-Restingstate_run-01_desc-iclabel_components.tsv
   Components:   63
   Excluded:     1

Label counts:
label
brain            33
other            27
eye blink         2
channel noise     1

First 10 components:
 component     label  label_probability  exclude
         0 eye blink             0.9847     True
         1     brain             0.9988    False
         2     brain             0.9894    False
         3     brain             0.8304    False
         4     brain             0.8921    False
         5     brain             0.9993    False
         6     brain             0.9006    False
         7 eye blink             0.4388    False
         8     other             0.6484    False
         9     brain             0.7076    False


Inspect the **cleaned EEG** file. It should:
- Open without error
- Have the same channel count as the input
- Have the same duration as the input
- Be roughly the same on-disk size as the Stage 1 input (the cleaning operation doesn't shrink the time series, just modifies it)

In [45]:
test_out_clean = cleaned_eeg_path(test_item[2])
if test_out_clean.exists():
    raw_check = mne.io.read_raw_fif(test_out_clean, preload=False, verbose=False)
    print(f'✅ Cleaned EEG: {test_out_clean}')
    print(f'   Channels:     {len(raw_check.ch_names)}')
    print(f'   Duration:     {raw_check.times[-1]:.1f} s')
    print(f'   Sample rate:  {raw_check.info["sfreq"]} Hz')
    print(f'   File size:    {test_out_clean.stat().st_size / 1e6:.1f} MB')
else:
    print(f'❌ No cleaned EEG — check {ERROR_LOG}')

✅ Cleaned EEG: /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-ica-apply/sub-10025/eeg/sub-10025_task-Restingstate_run-01_desc-clean_eeg.fif
   Channels:     68
   Duration:     63.0 s
   Sample rate:  250.0 Hz
   File size:    4.3 MB


---
# 7. Run the full batch

ICLabel runs ONNX inference (typically <5 s per file). Most of the wall time is `ica.apply()` plus GCS download/upload. Bumping `N_JOBS_OUTER` helps if you have spare RAM and bandwidth.

**The batch is resumable** — if it crashes mid-run, you can re-run this cell and it will skip files that already have both outputs.

In [ ]:
print(f'🚀 Stage 3 batch on {len(inputs)} files')
print(f'   Source:               {INPUT_SOURCE}')
print(f'   Upload to GCS:        {UPLOAD_TO_GCS}')
print(f'   Drop labels:          {sorted(EXCLUDE_LABELS)}')
print(f'   Probability threshold: {LABEL_PROB_THRESHOLD}')
print(f'   N_JOBS_OUTER:         {N_JOBS_OUTER}')
print(f'   Output:               {DERIV_ROOT}')
print(f'   Log:                  {LOG_FILE}\n')

if N_JOBS_OUTER == 1:
    for item in tqdm(inputs, desc='ICA label+apply'):
        process_one(item)
else:
    Parallel(n_jobs=N_JOBS_OUTER, verbose=10)(
        delayed(process_one)(item) for item in inputs
    )

print('\n✅ Batch complete.')

🚀 Stage 3 batch on 2356 files
   Source:               gcs
   Upload to GCS:        True
   Drop labels:          ['channel noise', 'eye blink', 'heart beat', 'line noise', 'muscle artifact']
   Probability threshold: 0.8
   N_JOBS_OUTER:         1
   Output:               /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-ica-apply
   Log:                  /Users/alirezafatemi/asd_eeg_pipeline/derivatives/logs/03_ica_label_apply_log.csv



ICA label+apply:   0%|          | 0/2356 [00:00<?, ?it/s]

/var/folders/83/_ppdmr2x49b7rpvgvxpwsfs40000gn/T/ipykernel_29292/3640590212.py:22: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  proba_matrix = iclabel_label_components(raw, ica)
/var/folders/83/_ppdmr2x49b7rpvgvxpwsfs40000gn/T/ipykernel_29292/2957725930.py:2: RuntimeWarning: Invalid tag with only 0/16 bytes at position 0 in file /tmp/stage3_cache/sub-10310_task-motor_run-03_desc-preproc_eeg.fif
  return mne.io.read_raw_fif(fpath, preload=True, verbose=False)
/var/folders/83/_ppdmr2x49b7rpvgvxpwsfs40000gn/T/ipykernel_29292/3640590212.py:22: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_refere

---
# 8. Inspect the run

Read the log fresh from disk and check three things:
1. Status counts (any failures?)
2. Distribution of component labels across the dataset
3. Files with unusual exclusion rates (zero ICs dropped, or >50% dropped)

In [ ]:
log_df = pd.read_csv(LOG_FILE)
print('Status counts:')
print(log_df['status'].value_counts(), '\n')

fails = log_df[log_df['status'] == 'fail']
if len(fails):
    print(f'⚠️  {len(fails)} failures — see {ERROR_LOG}')
    print(fails[['subject', 'task', 'run', 'error']].head(20).to_string(index=False))
else:
    print('🎉 No failures.')

**Component label distribution across the dataset.**  
Sanity expectations for a typical adult EEG dataset:
- `n_brain` is usually the largest count (often 40–60% of components)
- `n_eye` is typically 1–4 per recording
- `n_muscle` and `n_other` vary a lot
- `n_heart`, `n_line`, `n_channel` are often 0–2 each

If `n_brain` is suspiciously low across the board, your ICA fit may have problems — check Stage 2's convergence diagnostics.

In [ ]:
ok = log_df[log_df['status'] == 'ok']
if len(ok):
    print(f'Across {len(ok)} successful files:\n')

    print('Components / excluded / fraction excluded:')
    print(ok[['n_components', 'n_excluded', 'frac_excluded']].describe(), '\n')

    print('Mean count of each label per file:')
    label_cols = ['n_brain', 'n_other', 'n_muscle', 'n_eye',
                  'n_heart', 'n_line', 'n_channel']
    print(ok[label_cols].mean().round(2).to_string())

**Files with unusual exclusion rates.** Either flag triggers human review:
- `n_excluded == 0` → ICLabel found nothing artifact-y at the threshold; could be a clean recording, or could mean something is wrong with the IC topographies.
- `frac_excluded > 50%` → ICA likely failed to separate brain from artifacts.

In [ ]:
if len(ok):
    suspicious = ok[(ok['frac_excluded'] > MAX_BAD_FRACTION_WARN) |
                    (ok['n_excluded'] == 0)]
    if len(suspicious):
        print(f'⚠️  {len(suspicious)} files with unusual exclusion rates:')
        print(suspicious[['subject', 'task', 'run',
                          'n_components', 'n_excluded',
                          'frac_excluded', 'warning']]
              .head(20).to_string(index=False))
    else:
        print('✅ No files with unusual exclusion rates.')

**Per-subject summary** — useful for spotting subjects whose recordings consistently look weird (worth a deeper manual review).

In [ ]:
if len(ok):
    by_subject = (
        ok.groupby('subject')[['n_components', 'n_excluded', 'frac_excluded']]
          .agg(['mean', 'std'])
          .round(2)
    )
    print('Per-subject exclusion stats (first 20 subjects):')
    print(by_subject.head(20))

---
## ✅ What you have at the end of Stage 3

For every recording in the dataset:
- A **cleaned, average-referenced EEG** file (`desc-clean_eeg.fif`)
- A **per-component label TSV** (`desc-iclabel_components.tsv`) with full probability vectors — auditable and re-usable
- All artifacts uploaded to `gs://asd-eeg-dataset/derivatives/ds006780/mne-ica-apply/`

## Coming next: Stage 4 (Epoching + connectivity)

1. **Epoch the data** — task-locked epochs for FAST/IC/motor, fixed-length for Restingstate. AutoReject for final epoch-level rejection.
2. **Connectivity / PLV** — phase-locking value across electrode pairs in your bands of interest (theta, alpha, beta, gamma). `mne_connectivity.spectral_connectivity_epochs`.
3. **Group-level features** — band power, PLV matrices, graph metrics; merge with `participants.tsv` and demographic covariates.

Because Stage 3 outputs are already average-referenced and ICA-cleaned, Stage 4 can load `_desc-clean_eeg.fif` and go straight to epoching — no further preprocessing needed.